# 🧠 Huấn Luyện AI Cờ Tướng (Xiangqi HalfKP NNUE)
### Kết nối Engine Pikafish & Huấn luyện Kỳ Phổ Đại Sư

Notebook này hướng dẫn và tự động hóa toàn bộ quy trình huấn luyện cho dự án Game Cờ Tướng:
1. **Tải & Cài đặt Engine ngoài**: Tải và thiết lập Engine cờ tướng hàng đầu thế giới (**Pikafish**) vào thư mục biệt lập `engines/` (được `.gitignore` bảo vệ).
2. **Sinh Dữ Liệu Tự Động (Arena Data Generation)**: Tự động cho Pikafish thi đấu đối kháng (Self-play) ở độ sâu cao để sinh dữ liệu kỳ phổ thế cờ phong phú.
3. **Huấn luyện Giám sát (Supervised Learning)**: Huấn luyện mạng NNUE HalfKP từ tập kỳ phổ mẫu danh thủ ở định dạng Float nguyên bản (`.f32`).
4. **Chưng cất Tri thức (Knowledge Distillation)**: Kế thừa mạng Float từ bước trước, dùng Pikafish làm "Sư phụ" đánh giá và gán nhãn tinh chỉnh thế cờ.
5. **Đấu tập Đối kháng (Sparring)**: Cho Bot với mô hình Float thi đấu trực tiếp với Engine ngoài để đo lường trình độ thực chiến.
6. **Lượng tử hóa Mô hình (Quantization)**: Chuyển đổi mô hình Float f32 sang Mô Hình NNUE Chuẩn SIMD và lưu vào `cotuong_core/assets/default.nnue`.
7. **Kiểm thử & Đóng gói WASM**: Xác minh tính toàn vẹn và tự động biên dịch lại gói WebAssembly (`cotuong_core.wasm`) cho giao diện web.

> 💡 **Độ Chính Xác Tuyệt Đối (No Intermediate Quantization Loss)**: Toàn bộ quá trình huấn luyện và kế thừa giữa các bước đều sử dụng trọng số Float 32-bit (`checkpoints/model_latest.f32`), loại bỏ hoàn toàn sai số tích lũy do lượng tử hóa sớm. Chỉ bước cuối cùng mới thực hiện lượng tử hóa.

---
## 🛠️ Bước 1: Kiểm Tra Môi Trường & Khởi Tạo Thư Mục Làm Việc


In [8]:
# Tạo các thư mục lưu trữ riêng (đã được cấu hình trong .gitignore để không đẩy lên GitHub)
!mkdir -p engines checkpoints

import os
import sys
import subprocess

print("🔍 Kiểm tra môi trường hệ thống:")
!rustc --version
!cargo --version
!node --version
!npm --version

cpu_cores = os.cpu_count()
print(f"✅ Số nhân CPU khả dụng: {cpu_cores}")
print(f"📁 Thư mục Engines    : {os.path.abspath('engines')}")
print(f"📁 Thư mục Checkpoints: {os.path.abspath('checkpoints')}")


🔍 Kiểm tra môi trường hệ thống:
rustc 1.96.0 (ac68faa20 2026-05-25)
cargo 1.96.0 (30a34c682 2026-05-25)
v22.22.2
10.9.7
✅ Số nhân CPU khả dụng: 16
📁 Thư mục Engines    : /home/exblackhole/Desktop/GameCoTuong/engines
📁 Thư mục Checkpoints: /home/exblackhole/Desktop/GameCoTuong/checkpoints


---
## 🐟 Bước 2: Tải & Cài Đặt Engine Pikafish (UCCI / UCI Protocol)
**Pikafish** là engine cờ tướng mạnh nhất thế giới hiện nay (dựa trên nền tảng Stockfish). Ở bước này, chúng ta sẽ thiết lập binary Pikafish vào thư mục `engines/pikafish/`.


In [9]:
%%bash
set -e
ENGINES_DIR="$(pwd)/engines"
PIKAFISH_DIR="$ENGINES_DIR/pikafish"
mkdir -p "$PIKAFISH_DIR"

TARGET_BIN="$PIKAFISH_DIR/pikafish"

if [ -f "$TARGET_BIN" ]; then
    echo "✅ Pikafish đã có sẵn tại: $TARGET_BIN"
else
    echo "⏳ Đang chuẩn bị Engine Pikafish..."
    cd "$PIKAFISH_DIR"
    
    DOWNLOAD_SUCCESS=0
    RELEASE_URL=$(curl -s https://api.github.com/repos/official-pikafish/Pikafish/releases/latest | grep "browser_download_url.*pikafish-.*x86-64.*\.zip" | head -n 1 | cut -d '"' -f 4 || true)
    
    if [ -n "$RELEASE_URL" ]; then
        echo "⬇️ Đang tải bản phát hành từ: $RELEASE_URL"
        curl -L -o pikafish_release.zip "$RELEASE_URL"
        unzip -o pikafish_release.zip
        rm -f pikafish_release.zip
        FOUND_BIN=$(find . -maxdepth 2 -type f -name "pikafish*" ! -name "*.zip" ! -name "*.nnue" | head -n 1)
        if [ -n "$FOUND_BIN" ]; then
            chmod +x "$FOUND_BIN"
            cp "$FOUND_BIN" "$TARGET_BIN"
            DOWNLOAD_SUCCESS=1
        fi
    fi
    
    if [ "$DOWNLOAD_SUCCESS" -eq 0 ]; then
        echo "🛠️ Đang biên dịch Pikafish từ nguồn..."
        rm -rf /tmp/pikafish_src
        git clone --depth 1 https://github.com/official-pikafish/Pikafish.git /tmp/pikafish_src
        make -C /tmp/pikafish_src/src -j$(nproc) build ARCH=x86-64-modern
        cp /tmp/pikafish_src/src/pikafish "$TARGET_BIN"
        chmod +x "$TARGET_BIN"
        rm -rf /tmp/pikafish_src
    fi
fi

echo "🧪 Kiểm thử giao thức UCCI với Pikafish..."
echo -e "ucci\nquit" | "$TARGET_BIN" | head -n 6
echo "✅ Pikafish đã sẵn sàng tại: $TARGET_BIN"


✅ Pikafish đã có sẵn tại: /home/exblackhole/Desktop/GameCoTuong/engines/pikafish/pikafish
🧪 Kiểm thử giao thức UCCI với Pikafish...
Pikafish dev-20261029-fec5d430 by the Pikafish developers (see AUTHORS file)
Unknown command: 'ucci'. Type help for more information.
✅ Pikafish đã sẵn sàng tại: /home/exblackhole/Desktop/GameCoTuong/engines/pikafish/pikafish


---
## ⚔️ Bước 3: Sinh Dữ Liệu Thi Đấu Tự Động bằng Pikafish (Arena Data Generation)
Khi tập dữ liệu mẫu ban đầu còn ít, ta cho model AI hàng đầu thế giới **Pikafish** tự động đấu cờ với chính nó (Self-play) ở độ sâu cao (**depth 15**) qua các biến thể khai cuộc phong phú để sinh hàng nghìn ván cờ và thế cờ chất lượng cao (Master FEN Evaluations):

- **Kỳ phổ mẫu chuẩn cố định**: Được lưu riêng tại `cotuong_trainer/data/curated_master_games.txt` (15 đại danh cục kinh điển & các thế cờ FEN chuẩn).
- **Lưu trữ độc lập từng phiên sinh dữ liệu**: Mỗi lần sinh sẽ lưu thành một file riêng trong thư mục `cotuong_trainer/data/generated/` (đã được cấu hình `.gitignore` để không up lên GitHub).
- **Tập dữ liệu huấn luyện tổng hợp**: Kết quả sinh ra sẽ tự động gắn tiếp vào `cotuong_trainer/data/sample_master_games.txt` trên nền kỳ phổ mẫu chuẩn để phục vụ huấn luyện Supervised Learning.

In [11]:
%%bash
set -e

DATA_DIR="cotuong_trainer/data"
CURATED_FILE="$DATA_DIR/curated_master_games.txt"
SAMPLE_FILE="$DATA_DIR/sample_master_games.txt"
GENERATED_DIR="$DATA_DIR/generated"

mkdir -p "$GENERATED_DIR"

# 1. Đặt tên file riêng biệt cho phiên sinh dữ liệu (kèm timestamp, không up GitHub)
SESSION_FILE="$GENERATED_DIR/arena_$(date +%Y%m%d_%H%M%S).txt"
echo "🎮 Bắt đầu phiên sinh dữ liệu Arena -> Lưu tại: $SESSION_FILE"

# 2. Cho Pikafish tự thi đấu đối kháng (Self-Play / Arena)
cargo run --release -p cotuong_trainer -- \
    --mode gendata \
    --engine1 engines/pikafish/pikafish \
    --engine2 engines/pikafish/pikafish \
    --games 20 \
    --depth 15 \
    --dataset "$SESSION_FILE"

# 3. Khởi tạo sample_master_games.txt từ kỳ phổ mẫu chuẩn nếu chưa tồn tại
if [ ! -f "$SAMPLE_FILE" ]; then
    echo "📋 Đang sao chép kỳ phổ mẫu chuẩn $CURATED_FILE sang $SAMPLE_FILE..."
    cp "$CURATED_FILE" "$SAMPLE_FILE"
fi

# 4. Gắn tiếp kết quả ván đấu vừa sinh vào sample_master_games.txt
if [ -f "$SESSION_FILE" ] && [ -s "$SESSION_FILE" ]; then
    echo "📥 Đang gắn kết quả sinh dữ liệu vào $SAMPLE_FILE..."
    echo "" >> "$SAMPLE_FILE"
    cat "$SESSION_FILE" >> "$SAMPLE_FILE"
    echo "✅ Đã gộp thành công! Tổng số dòng trong $SAMPLE_FILE: $(wc -l < "$SAMPLE_FILE")"
    echo "📁 File phiên dữ liệu gốc được lưu độc lập tại: $SESSION_FILE [được gitignore]"
fi

🎮 Bắt đầu phiên sinh dữ liệu Arena -> Lưu tại: cotuong_trainer/data/generated/arena_20261107_184523.txt


   Compiling cotuong_trainer v0.1.0 (/home/exblackhole/Desktop/GameCoTuong/cotuong_trainer)
    Finished `release` profile [optimized] target(s) in 7.07s
     Running `target/release/cotuong_trainer --mode gendata --engine1 engines/pikafish/pikafish --engine2 engines/pikafish/pikafish --games 20 --depth 15 --dataset cotuong_trainer/data/generated/arena_20261107_184523.txt`


 🧠 Xiangqi AI Trainer (Master Dataset & External UCCI/UCI Engine Integration)  
Cấu hình:
  • Chế độ (Mode)            : gendata
  • Số Epochs                : 5
  • Tốc độ học (LR)          : 0.01
  • Đường dẫn trọng số đích : checkpoints/model_latest.f32
  • Kế thừa trọng số         : Tự động từ checkpoints/model_latest.f32 (nếu có)
  • Thư mục Checkpoint       : checkpoints

▶ Khởi tạo Đấu Trường Sinh Dữ Liệu (Arena Engine Match):
   • Engine 1: engines/pikafish/pikafish (Uci)
   • Engine 2: engines/pikafish/pikafish (Ucci)
   • Số ván đấu: 20
   • Độ sâu phân tích mỗi nước: 15
   • Tệp lưu dữ liệu: cotuong_trainer/data/generated/arena_20261107_184523.txt
✅ Đã kết nối thành công với cả 2 Engine!

▶ [Ván 1/20] Engine 1 (ĐỎ) vs Engine 2 (ĐEN)
   🏆 Thắng áp đảo (Adjudication bởi Eval 1345) -> Kết thúc ván tại nước 74.
   🏁 Kết quả ván 1: 1-0 (Tổng 74 nước đi, trích xuất 71 thế cờ FEN)

▶ [Ván 2/20] Engine 2 (ĐỎ) vs Engine 1 (ĐEN)
   🏆 Thắng áp đảo (Adjudication bởi Eval 10000) -> Kết t

---
## 📖 Bước 4: Huấn Luyện Giám Sát Làm Mới từ Kỳ Phổ Mẫu (Supervised Learning - Scratch)
Huấn luyện mạng NNUE học trực tiếp từ các thế cờ danh thủ trong `cotuong_trainer/data/sample_master_games.txt`.

- **Khởi tạo làm mới (`--scratch`)**: Mỗi lần chạy lại notebook, mô hình sẽ luôn được huấn luyện mới từ đầu để đảm bảo trọng số sạch, chuẩn xác và không bị nhiễm lỗi từ các lần chạy trước.
- **Định dạng**: Trọng số lưu ở dạng Float 32-bit (`checkpoints/model_latest.f32`) chưa lượng tử hóa để đảm bảo độ chính xác tối đa.
- **Checkpoints**: Lưu từng thế hệ vào `checkpoints/dataset_epoch_{N}.f32`.

In [12]:
# Chạy huấn luyện Supervised Learning làm mới từ đầu (--scratch)
!cargo run --release -p cotuong_trainer -- \
    --mode dataset \
    --scratch \
    --epochs 20 \
    --lr 0.01 \
    --eval-games 10 \
    --checkpoints checkpoints \
    --output checkpoints/model_latest.f32

    Finished `release` profile [optimized] target(s) in 0.03s
     Running `target/release/cotuong_trainer --mode dataset --scratch --epochs 20 --lr 0.01 --eval-games 10 --checkpoints checkpoints --output checkpoints/model_latest.f32`
 🧠 Xiangqi AI Trainer (Master Dataset & External UCCI/UCI Engine Integration)  
Cấu hình:
  • Chế độ (Mode)            : dataset
  • Số Epochs                : 20
  • Tốc độ học (LR)          : 0.01
  • Đường dẫn trọng số đích : checkpoints/model_latest.f32
  • Huấn luyện mới từ đầu    : BẬT (--scratch)
  • Thư mục Checkpoint       : checkpoints
  • Tập dữ liệu kỳ phổ mẫu   : cotuong_trainer/data/sample_master_games.txt
  • Số ván Đấu Đánh Giá      : 10

▶ Nạp dữ liệu kỳ phổ mẫu từ: cotuong_trainer/data/sample_master_games.txt ...
  • Đã nạp thành công 7267 thế cờ mẫu!
🌱 Khởi tạo mô hình NNUE mới từ đầu (--scratch)...

--------------------------------------------------------------------------------
 🏆 EPOCH 1/20 (SUPERVISED LEARNING TỪ KỲ PHỔ)
-----------

---
## 👨‍🏫 Bước 5: Chưng Cất Tri Thức cùng Sư Phụ Pikafish (Knowledge Distillation)
Kế thừa trọng số Float f32 từ Bước 4 (`checkpoints/model_latest.f32`), dùng Pikafish làm "Sư phụ" để gán nhãn điểm số (`search_eval` ở độ sâu 10–14) và tiếp tục tối ưu hóa mạng ở độ chính xác Float f32. Chế độ này chỉ sinh checkpoint duy nhất ở epoch cuối cùng với tên file đặt theo thời điểm huấn luyện xong: `teacher_YYYYMMDD_HHMMSS.f32`.


In [13]:
# Chạy chưng cất tri thức với Engine sư phụ Pikafish (Kế thừa Float f32 model)
!cargo run --release -p cotuong_trainer -- \
    --mode teacher \
    --engine engines/pikafish/pikafish \
    --proto uci \
    --depth 15 \
    --epochs 100\
    --lr 0.01 \
    --checkpoints checkpoints \
    --output checkpoints/model_latest.f32


    Finished `release` profile [optimized] target(s) in 0.03s
     Running `target/release/cotuong_trainer --mode teacher --engine engines/pikafish/pikafish --proto uci --depth 15 --epochs 100 --lr 0.01 --checkpoints checkpoints --output checkpoints/model_latest.f32`
 🧠 Xiangqi AI Trainer (Master Dataset & External UCCI/UCI Engine Integration)  
Cấu hình:
  • Chế độ (Mode)            : teacher
  • Số Epochs                : 100
  • Tốc độ học (LR)          : 0.01
  • Đường dẫn trọng số đích : checkpoints/model_latest.f32
  • Kế thừa trọng số         : Tự động từ checkpoints/model_latest.f32 (nếu có)
  • Thư mục Checkpoint       : checkpoints
  • Engine Sư Phụ            : engines/pikafish/pikafish (Uci)
  • Độ sâu phân tích         : 15
  • Tập thế cờ gán nhãn      : cotuong_trainer/data/sample_master_games.txt

▶ Khởi động Engine Sư Phụ ("engines/pikafish/pikafish", protocol: Uci) ...
✅ Đã kết nối thành công với Engine sư phụ qua giao thức chuẩn!
▶ Nạp tập thế cờ cần gán nhãn từ: cotu

---
## 🥊 Bước 6: Đấu Tập Đối Kháng Trực Tiếp & Tự Động Sửa Sai (Sparring & Online Learning)
Bot thi đấu luân phiên Đỏ / Đen với Đại sư phụ Pikafish.

- **Vừa đấu vừa học (Online Learning)**: Sau mỗi ván đấu, hệ thống lập tức trích xuất toàn bộ thế cờ và điểm số đánh giá từ Pikafish trong ván đó để huấn luyện sửa sai ngay 1 đợt.
- **Tiến bộ từng ván**: Trọng số mô hình được cập nhật ngay lập tức trước khi bước vào ván đấu tiếp theo, giúp Pikafish nắn chỉnh Bot dần dần qua từng ván cờ.
- **Lưu trữ**: Mô hình sau khi tôi luyện được tự động lưu lại vào `checkpoints/model_latest.f32`.

In [14]:
# Đấu tập và tự động sửa sai theo từng ván với Pikafish ở độ sâu 15
!cargo run --release -p cotuong_trainer -- \
    --mode sparring \
    --engine engines/pikafish/pikafish \
    --proto uci \
    --games 100 \
    --depth 15 \
    --resume checkpoints/model_latest.f32

    Finished `release` profile [optimized] target(s) in 0.02s
     Running `target/release/cotuong_trainer --mode sparring --engine engines/pikafish/pikafish --proto uci --games 100 --depth 15 --resume checkpoints/model_latest.f32`
 🧠 Xiangqi AI Trainer (Master Dataset & External UCCI/UCI Engine Integration)  
Cấu hình:
  • Chế độ (Mode)            : sparring
  • Số Epochs                : 5
  • Tốc độ học (LR)          : 0.01
  • Đường dẫn trọng số đích : checkpoints/model_latest.f32
  • Nạp trọng số khởi điểm  : checkpoints/model_latest.f32
  • Thư mục Checkpoint       : checkpoints
  • Engine Đối thủ           : engines/pikafish/pikafish (Uci)
  • Số ván đấu               : 100
  • Độ sâu đấu tập           : 15

▶ Khởi tạo đấu tập & tự động sửa sai (Sparring & Online Learning) với: engines/pikafish/pikafish ...
✅ Đã sẵn sàng thi đấu 100 ván đối kháng (vừa đấu vừa học từ Pikafish)!
♻️  Đã nạp thành công trọng số từ: checkpoints/model_latest.f32 để tiếp tục huấn luyện!

  ▶ [Ván 1/100

---
## 🎯 Bước 7: Lượng Tử Hóa Sang Mô Hình NNUE Chuẩn vào cotuong_core/assets (default.nnue)
Chế độ `--mode quantize` chuyển đổi mô hình huấn luyện Float f32 (`checkpoints/model_latest.f32`) sang định dạng Mô Hình NNUE Chuẩn (Stockfish / Pikafish NNUE):
1. **Mô Hình NNUE Chuẩn (SIMD i16/i8)** (`cotuong_core/assets/default.nnue` ~4.0 MB): Trọng số Feature Transformer 16-bit, SCReLU, các tầng FC i8/i16, thuần túy số nguyên, nhúng trực tiếp trong WebAssembly Engine.
2. **Mô Hình Không Lượng Tử (Float 32-bit)** (`cotuong_core/assets/model_f32.bin` ~8.1 MB): Lưu trữ độ chính xác nguyên bản để tiếp tục huấn luyện tăng cường (fine-tuning).


In [1]:
# Lượng tử hóa mô hình sang chuẩn NNUE (Dual-Head HalfKP v3 SIMD) nhúng sẵn vào cotuong_core/assets
!cargo run --release -p cotuong_trainer -- \
    --mode quantize \
    --input checkpoints/model_latest.f32 \
    --output cotuong_core/assets/default.nnue


   Compiling cotuong_core v0.1.0 (/home/exblackhole/Desktop/GameCoTuong/cotuong_core)
   Compiling cotuong_trainer v0.1.0 (/home/exblackhole/Desktop/GameCoTuong/cotuong_trainer)
    Finished `release` profile [optimized] target(s) in 8.47sr(bin)        
     Running `target/release/cotuong_trainer --mode quantize --input checkpoints/model_latest.f32 --output cotuong_core/assets/default.nnue`
 🧠 Xiangqi AI Trainer (Master Dataset & External UCCI/UCI Engine Integration)  
Cấu hình:
  • Chế độ (Mode)            : quantize
  • File Float nguồn         : checkpoints/model_latest.f32
  • File NNUE đích           : cotuong_core/assets/default.nnue

▶ Bắt đầu lượng tử hóa sang Mô Hình NNUE Chuẩn (Dual-Head HalfKP v3)...
  • File nguồn (Float NNUE)    : checkpoints/model_latest.f32
  1️⃣  Không lượng tử (Float 32-bit)   : cotuong_core/assets/model_f32.bin (8.09 MB)
  2️⃣  Mô Hình NNUE Chuẩn (SIMD i16/i8): cotuong_core/assets/default.nnue (4.03 MB)
✅ Đã lượng tử hóa thành công mô hình NNUE chuẩn

---
## 🗄️ Bước 8: Quản Lý Checkpoints, Kiểm Thử & Đóng Gói WebAssembly (WASM)
1. Liệt kê các checkpoint Float (`.f32`) và kiểm tra dung lượng tệp lượng tử hóa `default.nnue`.
2. Kiểm thử tính hợp lệ của trọng số nhúng và Accumulator trong `cotuong_core`.
3. Tự động biên dịch lại gói WebAssembly (`cotuong_core.wasm`) sẵn sàng cho giao diện web.


In [2]:
import glob
import os

print("📦 Danh sách các Checkpoints Float đã lưu trong checkpoints/:")
float_checkpoints = sorted(glob.glob("checkpoints/*.f32"))
if float_checkpoints:
    for cp in float_checkpoints:
        size_mb = os.path.getsize(cp) / (1024.0 * 1024.0)
        print(f"  • {cp} ({size_mb:.2f} MB)")
else:
    print("  (Chưa có file checkpoint float nào, hãy chạy bước huấn luyện ở trên)")

print("\n▶ Trọng số chính đã lượng tử hóa:")
default_nnue = "cotuong_core/assets/default.nnue"
if os.path.exists(default_nnue):
    print(f"  • {default_nnue} ({os.path.getsize(default_nnue) / 1024.0:.1f} KB)")


📦 Danh sách các Checkpoints Float đã lưu trong checkpoints/:
  • checkpoints/dataset_epoch_1.f32 (8.09 MB)
  • checkpoints/dataset_epoch_10.f32 (8.09 MB)
  • checkpoints/dataset_epoch_11.f32 (8.09 MB)
  • checkpoints/dataset_epoch_12.f32 (8.09 MB)
  • checkpoints/dataset_epoch_13.f32 (8.09 MB)
  • checkpoints/dataset_epoch_14.f32 (8.09 MB)
  • checkpoints/dataset_epoch_15.f32 (8.09 MB)
  • checkpoints/dataset_epoch_16.f32 (8.09 MB)
  • checkpoints/dataset_epoch_17.f32 (8.09 MB)
  • checkpoints/dataset_epoch_19.f32 (8.09 MB)
  • checkpoints/dataset_epoch_2.f32 (8.09 MB)
  • checkpoints/dataset_epoch_20.f32 (8.09 MB)
  • checkpoints/dataset_epoch_4.f32 (8.09 MB)
  • checkpoints/dataset_epoch_5.f32 (8.09 MB)
  • checkpoints/dataset_epoch_6.f32 (8.09 MB)
  • checkpoints/dataset_epoch_7.f32 (8.09 MB)
  • checkpoints/dataset_epoch_8.f32 (8.09 MB)
  • checkpoints/dataset_epoch_9.f32 (8.09 MB)
  • checkpoints/model_latest.f32 (8.09 MB)
  • checkpoints/teacher_20261107_011233.f32 (8.09 MB)
  • 

In [3]:
# Kiểm thử tính hợp lệ và Accumulator của model NNUE vừa lượng tử hóa
!cargo test -p cotuong_core -- test_embedded_weights_validity test_accumulator_incremental_vs_refresh

# Biên dịch lại gói WebAssembly (WASM) với trọng số mới
!npm run build:wasm
print("\n🎉 Hoàn tất đóng gói! Bạn có thể chạy 'npm run dev' trên terminal để trải nghiệm AI trên trình duyệt.")


   Compiling cotuong_core v0.1.0 (/home/exblackhole/Desktop/GameCoTuong/cotuong_core)
    Finished `test` profile [unoptimized + debuginfo] target(s) in 0.29s   
     Running unittests src/lib.rs (target/debug/deps/cotuong_core-132c201bcd89cba7)

running 2 tests
test engine::nnue::tests::test_embedded_weights_validity ... ok
test engine::nnue::tests::test_accumulator_incremental_vs_refresh ... ok

test result: ok. 2 passed; 0 failed; 0 ignored; 0 measured; 35 filtered out; finished in 0.07s

     Running tests/mate_solving.rs (target/debug/deps/mate_solving-d1aa54a08addb97b)

running 0 tests

test result: ok. 0 passed; 0 failed; 0 ignored; 0 measured; 4 filtered out; finished in 0.00s

   Doc-tests cotuong_core

running 0 tests

test result: ok. 0 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 0.00s


> game-co-tuong@0.1.0 build:wasm
> cargo build -p cotuong_core --target wasm32-unknown-unknown --release && wasm-bindgen --target web --out-dir src/wasm target/wasm3